# 01 Data Loading

This notebook validates the raw-data integration phase for the four approved dataset categories only. It checks the environment, confirms approved raw folders, lists supported and unsupported raw files, previews discovered datasets safely, and regenerates inventory reports without starting EDA, preprocessing, HWI scoring, or model training.


In [ ]:
from __future__ import annotations

import platform
import sys
from pathlib import Path

def detect_project_root(start: Path | None = None) -> Path:
    start_path = (start or Path.cwd()).resolve()
    for candidate in (start_path, *start_path.parents):
        if (candidate / "src").exists() and (candidate / "data").exists():
            return candidate
    raise FileNotFoundError("Could not detect the project root.")

PROJECT_ROOT = detect_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project root: {PROJECT_ROOT}")
print(f"Python executable: {sys.executable}")
print(f"Python version: {sys.version.split()[0]}")
print(f"Platform: {platform.platform()}")


In [ ]:
import pandas as pd
from IPython.display import display

from src.config import (
    APPROVED_RAW_DATASET_CATEGORIES,
    DATASET_METADATA_PATH,
    RAW_DATA_DIR,
    REPORTS_DIR,
)
from src.data_loader import (
    build_dataset_inventory,
    create_raw_file_manifest,
    discover_dataset_files,
    find_unexpected_raw_categories,
    load_dataset_metadata,
    load_dataset_preview,
    resolve_dataset_metadata,
    validate_approved_raw_categories,
)

print(f"Raw data directory: {RAW_DATA_DIR}")
print(f"Reports directory: {REPORTS_DIR}")
print(f"Approved raw categories: {APPROVED_RAW_DATASET_CATEGORIES}")
print(f"Dataset metadata file: {DATASET_METADATA_PATH}")


In [ ]:
category_status = validate_approved_raw_categories(RAW_DATA_DIR)
unexpected_categories = [path.name for path in find_unexpected_raw_categories(RAW_DATA_DIR)]
manifest_df = create_raw_file_manifest(RAW_DATA_DIR)
supported_manifest_df = manifest_df[manifest_df["supported"] == True].copy() if not manifest_df.empty else pd.DataFrame()
unsupported_manifest_df = manifest_df[manifest_df["supported"] == False].copy() if not manifest_df.empty else pd.DataFrame()

display(pd.DataFrame([{"category": key, "exists": value} for key, value in category_status.items()]))
print(f"Unexpected raw categories: {unexpected_categories if unexpected_categories else "None"}")
print(f"Supported dataset files discovered: {len(supported_manifest_df)}")
print(f"Unsupported raw files discovered: {len(unsupported_manifest_df)}")

if not supported_manifest_df.empty:
    display(supported_manifest_df)
if not unsupported_manifest_df.empty:
    display(unsupported_manifest_df)


In [ ]:
metadata_map = load_dataset_metadata(DATASET_METADATA_PATH)
preview_rows = []
preview_failures = []

for dataset_path in discover_dataset_files(RAW_DATA_DIR):
    relative_path = dataset_path.relative_to(RAW_DATA_DIR).as_posix()
    metadata = resolve_dataset_metadata(dataset_path, metadata_map, raw_data_dir=RAW_DATA_DIR)
    try:
        preview_df = load_dataset_preview(dataset_path, preview_rows=5)
        preview_rows.append(
            {
                "relative_path": relative_path,
                "category": relative_path.split("/")[0],
                "extension": dataset_path.suffix.lower(),
                "size_mb": round(dataset_path.stat().st_size / (1024 * 1024), 4),
                "preview_row_count": len(preview_df),
                "column_names": list(preview_df.columns),
                "dtypes": {str(column): str(dtype) for column, dtype in preview_df.dtypes.items()},
                "candidate_target_suggestions": [
                    column
                    for column in preview_df.columns
                    if any(keyword in str(column).lower() for keyword in ("label", "target", "class", "type", "outcome", "phishing"))
                ],
                "unit_of_analysis_note": metadata.get("unit_of_analysis", "Requires verification"),
            }
        )
    except Exception as exc:
        preview_failures.append(
            {
                "relative_path": relative_path,
                "error": f"{type(exc).__name__}: {exc}",
            }
        )

if preview_rows:
    display(pd.DataFrame(preview_rows))
if preview_failures:
    print("Preview failures:")
    display(pd.DataFrame(preview_failures))


In [ ]:
inventory_df, quality_df, diagnostics = build_dataset_inventory(
    raw_data_dir=RAW_DATA_DIR,
    metadata_path=DATASET_METADATA_PATH,
    output_dir=REPORTS_DIR,
    return_diagnostics=True,
)

print("Inventory reports written to:")
print(f" - {REPORTS_DIR / "dataset_inventory.csv"}")
print(f" - {REPORTS_DIR / "dataset_inventory.json"}")
print(f" - {REPORTS_DIR / "data_quality_summary.csv"}")
print(f" - {REPORTS_DIR / "raw_file_manifest.csv"}")
print(f" - {REPORTS_DIR / "unsupported_raw_files.csv"}")
print(f"Successful dataset loads: {len(diagnostics.successful_files)}")
print(f"Failed dataset loads: {len(diagnostics.failed_files)}")

display(inventory_df.head())
display(quality_df.head())

if diagnostics.failed_files:
    display(pd.DataFrame([{"relative_path": key, "error": value} for key, value in diagnostics.failed_files.items()]))
